## 6. Why and when GMRES converges

*Analysis, no new version.* This step uses `gmres_v4` exactly as Step 5 left it. It doesn't change what GMRES computes or how. It asks how many steps GMRES needs, and why that number is 11 on one problem and $n$ on another.

**Where Step 5 left off.** We want to solve $Ax = b$ for an $n \times n$ matrix $A$ that we can only apply to vectors, through a function `matvec` $: v \mapsto Av$. Starting from $x_0 = 0$, GMRES picks at step $k$ the vector $x_k$ in the Krylov space

$$
\mathcal{K}_k = \operatorname{span}\{\, b,\ Ab,\ \dots,\ A^{k-1} b \,\}
$$

whose residual $r_k = b - A x_k$ has the smallest 2-norm. `gmres_v4` does that with an orthonormal basis built by Arnoldi (Step 2), a small $(k+1) \times k$ least-squares problem (Step 3) and Givens rotations (Step 4), and it stays safe when the Krylov space stops growing (Step 5). Step 4 found that each step multiplies the residual by a factor, $\|r_k\|_2 = |s_k|\, \|r_{k-1}\|_2$, where $s_k$ is the sine of the $k$-th Givens rotation. Section 4.7 measured it: $|s_k|$ is about 0.1 on `diag_narrow`, about 0.96 on `convdiff_n` (with velocity $c = 20$) and exactly 1 on `shift_n` until the last step. Nothing in Steps 1–5 says why.

**The plan for this step.** One equation, one inequality and one picture. The equation (6.1): the residual is a polynomial in $A$ applied to $b$, $r_k = q_k(A)\, b$, and that polynomial always has $q_k(0) = 1$. The inequality (6.2): GMRES's $q_k$ is at least as good as any polynomial we can write down. The picture, which Steps 7 and 8 reuse: the eigenvalues of $A$ drawn as dots on an axis, and $q_k$ drawn as a curve held at 1 at $z = 0$. GMRES is fast when a curve of low degree, held at that point, can be small on all the dots. The rest of the step applies the picture: bunches of eigenvalues at different distances from 0 (6.2), eigenvalues spread over decades down toward 0 (6.3), a few tight bunches (6.4), eigenvalues on a ring around 0 (6.5), and a matrix whose eigenvalues mislead, `convdiff_n` (6.6). Section 6.7 turns the knob that places the eigenvalues of an implicit time step, the time step $\Delta t$, and 6.8 counts what many steps cost.

### Setup

[tutorial_utils.py](tutorial_utils.py) holds what every step of the tutorial uses: `CountingMatvec` from Step 1, the example problems and the plot style, including `plot_residual_polynomial`, new in this step, which draws the dots-and-curve picture. Each example builder returns a pair `(A, b)`. As before, a solver only ever gets the matrix wrapped in a `CountingMatvec`. In this step we also use `A` directly, but only to look, never to solve: to form the residual $r_k = b - A x_k$ of an iterate, to compute eigenvalues for the plots, and to compute the **condition number** $\operatorname{cond}(A) = \|A\|_2\, \|A^{-1}\|_2$, the ratio of $A$'s largest to smallest singular value.

The example problems in this notebook, each a matrix $A$ with a right-hand side $b$:

- **`diag_narrow`**: the 100×100 diagonal matrix with eigenvalues evenly spaced from 1 to 1.5, $\lambda_i = 1 + 0.5\,(i-1)/99$, and $b = (1, \dots, 1)$. The easy case.
- **`diag_spread`**: the 20×20 diagonal matrix with eigenvalues evenly spaced on a log scale from 1 to $10^4$, $\lambda_i = 10^{4(i-1)/19}$, and $b = (1, \dots, 1)$. Condition number $10^4$.
- **`diag_clustered`** (new in this step, added to `tutorial_utils.py`): a diagonal matrix whose $n$ eigenvalues sit in three tight bunches, around 1, 2 and 3. The $n$ eigenvalues are shared between the bunches as evenly as possible, and each bunch is evenly spaced over $[\text{center} - 10^{-3},\ \text{center} + 10^{-3}]$. $b = (1, \dots, 1)$. Used with $n = 30$, and with $n = 3000$ to show that $n$ doesn't matter.
- **`shift_n`**: the 8×8 cyclic shift, $A e_j = e_{j+1}$ and $A e_8 = e_1$, with $b = e_1$, where $e_j$ is the $j$-th unit vector. Orthogonal, so its condition number is 1.
- **`convdiff_n`**: the 20×20 tridiagonal matrix from Step 3, section 3.3, which discretises $-u''(s) + c\, u'(s)$ on $[0, 1]$ with $u(0) = u(1) = 0$, on the 20 interior grid points $s_i = i\, \Delta s$ with spacing $\Delta s = 1/21$:
  $$
  \frac{2}{\Delta s^2} \text{ on the diagonal}, \qquad
  -\frac{1}{\Delta s^2} - \frac{c}{2 \Delta s} \text{ below it}, \qquad
  -\frac{1}{\Delta s^2} + \frac{c}{2 \Delta s} \text{ above it},
  $$
  with $b = (1, \dots, 1)$. The velocity $c$ now has a default, **$c = 20$** (new in this step; it is the value Steps 3–5 used). Section 6.6 also uses $c = 0$, where the matrix is the symmetric matrix of $-u''$ alone, and $c = 10$, 30 and 42. The number $c\, \Delta s / 2$, the **cell Péclet number**, compares advection with diffusion over one cell.
- **Built in this notebook** (not in `tutorial_utils.py`), each with $b = (1, \dots, 1)$ unless stated:
  - three **shifted bunches**, $A = \operatorname{diag}(\text{100 points evenly spaced in } [a,\ a + 0.5])$ for $a = 0.1$, 1 and 10 (6.2);
  - a **bunch with outliers**: the 100 eigenvalues of `diag_narrow` together with 5, 20 and 100, and separately together with 0.01 (6.4);
  - the **diagonal twin** of `convdiff_n`: the diagonal matrix with the same 20 eigenvalues as `convdiff_n` (6.6);
  - an **implicit time step**, $A = I + \Delta t\, L$, with $L$ = `convdiff_n(400, c=0)` and $b_i = s_i$ (`rhs="ramp"`) (6.7).

`%autoreload 2` picks up edits to `tutorial_utils.py` without restarting the kernel.

In [ ]:
%load_ext autoreload
%autoreload 2

import numpy as np  # noqa: I001
import matplotlib.pyplot as plt
from numpy.polynomial import Polynomial

from tutorial_utils import (
    # the matvec wrapper from Step 1
    CountingMatvec,
    # loading definitions from an earlier step's notebook
    import_from_notebook,
    # example problems; each returns (A, b)
    diag_narrow, diag_spread, diag_clustered, convdiff_n, shift_n,
    # plot style
    style_axes, plot_residuals, plot_matrix, plot_residual_polynomial, add_figure_caption,
    REFERENCE_COLOR, SERIES_COLORS, SEQUENTIAL_CMAP,
)

np.set_printoptions(precision=4, suppress=True)
bottom = 1e-17   # where an exact zero is drawn on a log axis

A_narrow, b_narrow = diag_narrow()                             # 100 x 100 diagonal, eigenvalues in [1, 1.5]
A_spread, b_spread = diag_spread()                             # 20 x 20 diagonal, eigenvalues from 1 to 1e4
A_clustered, b_clustered = diag_clustered()                    # 30 x 30 diagonal, three bunches (new)
A_clustered_large, b_clustered_large = diag_clustered(3000)    # the same bunches, 3000 x 3000 (72 MB)
A_shift, b_shift = shift_n(8)                                  # 8 x 8 cyclic shift, b = e_1
A_cd, b_cd = convdiff_n(20)                                    # 20 x 20 tridiagonal, c = 20 (the new default)
A_cd0, b_cd0 = convdiff_n(20, c=0.0)                           # the same grid with c = 0: symmetric

**From Step 5.** We need only `gmres_v4`, loaded from [GMRES_5_breakdown.ipynb](GMRES_5_breakdown.ipynb) with `import_from_notebook`, which runs only that notebook's imports and definitions (`gmres_v4??` shows its source). Two keys of its record matter here: `residual_history`, the list $[\|r_0\|_2, \|r_1\|_2, \dots]$, and `iterations`, the number of steps taken.

In [ ]:
# Built in Step 5. Loaded from that notebook, so it stays the only copy.
gmres_v4 = import_from_notebook("GMRES_5_breakdown.ipynb", "gmres_v4")

### The puzzle

Four problems from Steps 1–5, each run through `gmres_v4` with its default tolerance, $\text{rtol} = 10^{-10}$. Figure 6.1 shows the residual histories. Steps to the tolerance is a poor measure of speed when the problems have different sizes, since GMRES finishes in at most $n$ steps whatever happens (Step 1.2). So the table also gives the mean factor per step over the first 7 steps, $(\|r_7\|_2 / \|b\|_2)^{1/7}$, from a second run with $\text{rtol} = 0$ and `max_iterations=7`.

In [ ]:
puzzle_examples = [
    ("diag_narrow", A_narrow, b_narrow, SERIES_COLORS[0], "-"),
    ("diag_spread", A_spread, b_spread, SERIES_COLORS[1], "-"),
    ("convdiff_n, c = 20", A_cd, b_cd, SERIES_COLORS[2], "-"),
    ("shift_n", A_shift, b_shift, REFERENCE_COLOR, "--"),   # the reference worst case
]
fig, ax = plt.subplots(figsize=(9, 4.2), constrained_layout=True)
print(f"{'example':20s} {'n':>4s} {'cond(A)':>9s} {'steps to 1e-10':>15s} {'mean factor per step, k = 1..7':>31s}")
for name, A, b, color, linestyle in puzzle_examples:
    b_norm = np.linalg.norm(b)
    _, record = gmres_v4(CountingMatvec(A), b)
    _, record_7 = gmres_v4(CountingMatvec(A), b, rtol=0.0, max_iterations=7)
    factor = (record_7["residual_history"][-1] / b_norm) ** (1 / 7)
    plot_residuals(ax, np.maximum(record["residual_history"], bottom * b_norm), b_norm=b_norm, color=color,
                   linestyle=linestyle, label=f"{name}, n = {b.shape[0]}")
    print(f"{name:20s} {b.shape[0]:4d} {np.linalg.cond(A):9.3g} {record['iterations']:15d} {factor:31.3f}")
ax.axhline(1e-10, color=REFERENCE_COLOR, linewidth=1)
ax.text(12.5, 2e-10, "rtol", color=REFERENCE_COLOR, fontsize=9, va="bottom")
ax.text(7.25, 1e-4, "shift_n", color=REFERENCE_COLOR, fontsize=9, va="center")
ax.set_ylim(bottom / 2, 3)
ax.legend(frameon=False, loc="lower left", fontsize=9)
style_axes(ax, xticks=range(0, 21, 2))
add_figure_caption(fig, "6.1", r"Residual histories of four problems, rtol = $10^{-10}$. diag_narrow (n = 100) needs 11 steps; diag_spread and convdiff_n (n = 20) need all 20. shift_n (n = 8, grey, dashed) makes no progress until its last step. Exact zeros are drawn on the bottom edge.")
plt.show()

The table, read aloud. The condition numbers are 1.5, $10^4$, 50.8 and 1. The mean factors per step are 0.105, 0.930, 0.964 and 1.000. `diag_narrow` is the largest problem and by far the fastest. `shift_n` is the smallest, has the best possible condition number, 1, and is the slowest: it makes no progress at all for 7 steps. `diag_spread` has 200 times the condition number of `convdiff_n` and is faster per step. Neither the size $n$ nor $\operatorname{cond}(A)$ puts the four in order. What does?

### 6.1 The residual is a curve through (0, 1)

**The derivation.** Step 1.2 showed that every vector of $\mathcal{K}_k$ is a polynomial in $A$ applied to $b$: $x_k = p(A)\, b$ for some polynomial $p$ of degree less than $k$, where $p(A)$ means $p$'s powers of $z$ replaced by powers of $A$ (so $p(z) = 3 - 2z$ gives $p(A) = 3I - 2A$). Here $z$ is just the variable of the polynomials, a number. Then

$$
r_k = b - A\, p(A)\, b = q_k(A)\, b, \qquad q_k(z) = 1 - z\, p(z).
$$

$q_k$ is the **residual polynomial** of step $k$. It has degree at most $k$, and whatever $p$ is,

$$
q_k(0) = 1 .
$$

The converse holds too: a polynomial $q$ of degree at most $k$ with $q(0) = 1$ has $q(z) - 1$ zero at $z = 0$, so $q(z) - 1 = -z\, p(z)$ for a $p$ of degree less than $k$. So GMRES's rule, the smallest residual over $\mathcal{K}_k$, is the same as

$$
\|r_k\|_2 = \min_{\deg q \le k,\ q(0) = 1} \|q(A)\, b\|_2 ,
$$

and GMRES's $q_k$ is the polynomial that attains the minimum.

**Why $q(0) = 1$, in words.** At step 0 the residual is $b$ itself, so $q_0 = 1$. Every correction GMRES makes to $x$ changes the residual by $A$ times something, which adds only terms with at least one power of $z$. The constant term 1 never changes.

**For a diagonal $A$.** If $A = \operatorname{diag}(\lambda_1, \dots, \lambda_n)$, its eigenvalues are the $\lambda_i$, and $q(A) = \operatorname{diag}(q(\lambda_1), \dots, q(\lambda_n))$. So the $i$-th entry of the residual is

$$
r_k[i] = q_k(\lambda_i)\, b_i .
$$

With $b = (1, \dots, 1)$ the residual's entries *are* the heights $q_k(\lambda_i)$ of the curve at the eigenvalues, and

$$
\frac{\|r_k\|_2}{\|b\|_2} = \sqrt{\frac{1}{n} \sum_{i=1}^n q_k(\lambda_i)^2} ,
$$

the root-mean-square (RMS) height of the dots. GMRES picks, among all curves of degree at most $k$ through the point $(0, 1)$, the one with the smallest RMS height at the eigenvalues.

**Reading the curve off.** `residual_polynomial` below runs a solver for exactly $k$ steps ($\text{rtol} = 0$, `max_iterations=k`), forms $r_k$ with $A$ (looking only), and divides by $b$ to get the heights $r_k[i] / b_i = q_k(\lambda_i)$. Then it fits a polynomial of degree $k$ through the $n$ points $(\lambda_i, \text{height}_i)$ with NumPy's `Polynomial.fit`, a least-squares fit. The heights lie exactly on a polynomial of degree $k$, so the fit recovers $q_k$ up to round-off. The fit knows nothing about $q_k(0) = 1$, which makes $q_k(0) \approx 1$ a free check of it.

Figure 6.2 draws GMRES's $q_k$ on `diag_narrow` after 1, 2 and 3 steps, twice: $q_k$ on a linear axis, and $|q_k|$ on a log axis, where a root shows as a downward spike.

In [ ]:
def residual_polynomial(A, b, k, *, solver=gmres_v4, **solver_options):
    """Run a solver for k steps and read its residual polynomial q_k off the residual. Diagonal A only.

    For a diagonal A, r_k[i] = q_k(lambda_i) * b_i, so the heights r_k[i] / b_i are q_k
    at the eigenvalues, and a degree-k polynomial through those n points is q_k itself.
    Uses A directly, to look at the residual, never to solve.

    Args:
        A: a diagonal matrix.
        b: right-hand side with no zero entry, shape (n,).
        k: the number of steps, which is also the degree of the fitted q_k.
        solver: a solver with the shared signature (matvec, b, *, rtol, max_iterations, ...).
        **solver_options: passed on to the solver, e.g. restart=m for gmres_v5 (Step 7).

    Returns:
        eigenvalues (the diagonal of A), heights, q (a numpy Polynomial), record (the solver's).
    """
    eigenvalues = np.diag(A)
    assert np.array_equal(A, np.diag(eigenvalues)), "residual_polynomial needs a diagonal A"
    x, record = solver(CountingMatvec(A), b, rtol=0.0, max_iterations=k, **solver_options)
    heights = (b - A @ x) / b
    q = Polynomial.fit(eigenvalues, heights, deg=k)
    return eigenvalues, heights, q, record


fig, axes = plt.subplots(1, 2, figsize=(12, 4.2), constrained_layout=True)
for k, color in zip((1, 2, 3), SERIES_COLORS):
    eigenvalues, heights, q, record = residual_polynomial(A_narrow, b_narrow, k)
    for ax, magnitude in zip(axes, (False, True)):
        plot_residual_polynomial(ax, eigenvalues, heights, polynomial=q, magnitude=magnitude, color=color,
                                 label=f"k = {k}")
    print(f"k = {k}:  RMS height of the dots {np.sqrt(np.mean(heights**2)):.3g},  |q_k(0) - 1| = {abs(q(0.0) - 1):.1e},"
          f"  roots of q_k {np.sort(np.real_if_close(q.roots()))}")
axes[0].set_title(r"$q_k(z)$")
axes[1].set_title(r"$|q_k(z)|$, log scale")
axes[0].legend(frameon=False, loc="upper right")
for ax in axes:
    style_axes(ax)
add_figure_caption(fig, "6.2", r"GMRES's residual polynomial $q_k$ on diag_narrow after k = 1, 2, 3 steps. Dots: $r_k[i]/b_i$ at each eigenvalue $\lambda_i$. Open circle: the pinned point $q(0) = 1$. Left: $q_k$. Right: $|q_k|$ on a log scale, where each root is a downward spike.")
plt.show()

In Figure 6.2 each curve drops from 1 at $z = 0$ to near 0 across the bunch $[1, 1.5]$, and all $k$ of its roots lie inside the bunch: 1.27 for $k = 1$, then 1.11 and 1.40, then 1.06, 1.26 and 1.45. The RMS heights of the dots, 0.116, 0.0123 and 0.00127, are the relative residuals of section 4.7: about a factor 10 per step. The fitted curves pass $q_k(0) = 1$ to $6 \times 10^{-15}$ or better.

**The check.** For $k = 1, \dots, 8$: the RMS height of the dots against the residual the solver reports, `residual_history[-1]` $/ \|b\|_2$, and the fit's $|q_k(0) - 1|$.

In [ ]:
b_norm = np.linalg.norm(b_narrow)
print(f"{'k':>2s} {'RMS height of the dots':>23s} {'residual_history[-1] / ||b||':>29s} {'difference':>11s} {'|q_k(0) - 1|':>13s}")
for k in range(1, 9):
    eigenvalues, heights, q, record = residual_polynomial(A_narrow, b_narrow, k)
    rms = np.sqrt(np.mean(heights**2))
    reported = record["residual_history"][-1] / b_norm
    print(f"{k:2d} {rms:23.4e} {reported:29.4e} {abs(rms - reported):11.1e} {abs(q(0.0) - 1):13.1e}")
_, _, q, _ = residual_polynomial(A_narrow, b_narrow, 14)
print(f"\nk = 14, for comparison:  |q_14(0) - 1| = {abs(q(0.0) - 1):.1e}")

The two agree to round-off at every $k$, and every fitted curve passes $(0, 1)$. So **$\|r_k\|_2 / \|b\|_2$ is the RMS height of the dots**, and the curve through them is held at 1 at $z = 0$.

The curves in this step are drawn only for small $k$. The fit has to extrapolate from the dots to $z = 0$, and as $k$ grows that gets ill-conditioned: on `diag_narrow` the error of $q_k(0)$ is $3 \times 10^{-10}$ at $k = 8$, but $10^{-3}$ at $k = 14$. Each figure below prints $|q_k(0) - 1|$ as its check.

### 6.2 GMRES beats any curve you can draw

**A hand curve.** Every curve of degree at most $k$ through $(0, 1)$ is a candidate, so any one we can write down gives an upper bound on GMRES's residual. On `diag_narrow` take

$$
q(z) = \Big(1 - \frac{z}{1.25}\Big)^k ,
$$

of degree $k$, with $q(0) = 1$ and a $k$-fold root at the middle of the bunch. On $[1, 1.5]$, $|1 - z/1.25| = |z - 1.25| / 1.25 \le 0.25 / 1.25 = 0.2$, so every dot has height at most $0.2^k$, and so does their RMS.

**The bound.** GMRES picks the best curve, so on `diag_narrow`, $\|r_k\|_2 \le 0.2^k\, \|b\|_2$. The same argument works for any diagonal $A$ and any $b$: $\|q(A)\, b\|_2^2 = \sum_i q(\lambda_i)^2 b_i^2 \le \max_i q(\lambda_i)^2\, \|b\|_2^2$, so

$$
\|r_k\|_2 \le \max_i |q(\lambda_i)|\, \|b\|_2 \quad \text{for every } q \text{ with } \deg q \le k \text{ and } q(0) = 1 .
$$

GMRES is no worse than any curve through $(0, 1)$ of degree at most $k$.

**Aside.** The hand curve is an iteration of its own: $x \leftarrow x + \omega\, r$ with $\omega = 1/1.25$ multiplies the residual by $I - \omega A$ at every step, so its residual after $k$ steps is $(I - A/1.25)^k\, b$. This is Richardson iteration.

Figure 6.3 compares GMRES with the hand curve: the residual histories on the left, and the two curves after 5 steps on the right.

In [ ]:
hand_curve = Polynomial([1.0, -1 / 1.25])   # 1 - z / 1.25; the hand curve after k steps is hand_curve**k
eigenvalues = np.diag(A_narrow)
b_norm = np.linalg.norm(b_narrow)
_, record = gmres_v4(CountingMatvec(A_narrow), b_narrow, rtol=0.0, max_iterations=15)
steps = np.arange(16)
gmres_history = np.array(record["residual_history"]) / b_norm
hand_history = np.array([np.linalg.norm(hand_curve(eigenvalues) ** k * b_narrow) / b_norm for k in steps])

print("GMRES factor per step:        ", gmres_history[1:12] / gmres_history[:11])
print("hand curve's factor per step: ", hand_history[1:12] / hand_history[:11])
print(f"steps to 1e-10:  GMRES {np.argmax(gmres_history <= 1e-10)},  the bound 0.2^k {np.argmax(0.2**steps <= 1e-10)},"
      f"  the hand curve's actual residual {np.argmax(hand_history <= 1e-10)}")

fig, axes = plt.subplots(1, 2, figsize=(12, 4.2), constrained_layout=True)
ax = axes[0]
plot_residuals(ax, record["residual_history"], b_norm=b_norm, color=SERIES_COLORS[0], label="GMRES")
plot_residuals(ax, hand_history * b_norm, b_norm=b_norm, color=SERIES_COLORS[1], linestyle="--",
               label=r"hand curve $(1 - z/1.25)^k$: its actual residual")
ax.semilogy(steps, 0.2**steps, color=REFERENCE_COLOR, linewidth=1, label=r"the bound $0.2^k$")
ax.legend(frameon=False, loc="lower left", fontsize=9)
style_axes(ax, xticks=range(0, 16, 2))

ax = axes[1]
k = 5
eigenvalues, heights, q, _ = residual_polynomial(A_narrow, b_narrow, k)
hand = hand_curve**k
plot_residual_polynomial(ax, eigenvalues, heights, polynomial=q, magnitude=True, color=SERIES_COLORS[0],
                         label=f"GMRES's $q_{k}$")
plot_residual_polynomial(ax, eigenvalues, hand(eigenvalues), polynomial=hand, magnitude=True,
                         color=SERIES_COLORS[1], label=rf"hand curve $(1 - z/1.25)^{k}$")
roots = np.sort(np.real_if_close(q.roots()))
low = ax.get_ylim()[0]
ax.plot(roots, np.full(k, low * 3), linestyle="none", marker="^", markersize=8, color=SERIES_COLORS[0])
ax.plot([1.25], [low * 3], linestyle="none", marker="^", markersize=8, color=SERIES_COLORS[1])
ax.set_xlim(0.9, 1.6)
ax.set_title(f"k = {k}; triangles: the roots")
ax.legend(frameon=False, loc="upper right", fontsize=9)
style_axes(ax)
print(f"\nk = {k}:  GMRES's roots {roots};  |q_5(0) - 1| = {abs(q(0.0) - 1):.1e}")
print(f"         largest height on the dots: GMRES {np.abs(heights).max():.2e},  hand curve {np.abs(hand(eigenvalues)).max():.2e}")
print(f"         RMS height of the dots:     GMRES {np.sqrt(np.mean(heights**2)):.2e},  hand curve {np.sqrt(np.mean(hand(eigenvalues)**2)):.2e}")
add_figure_caption(fig, "6.3", r"Left: GMRES on diag_narrow against the hand curve $(1 - z/1.25)^k$, its bound $0.2^k$ and its actual residual. Right: both curves at k = 5 (z from 0.9 to 1.6). The hand curve stacks its five roots at 1.25; GMRES spreads them across the bunch, from 1.02 to 1.48, and its largest height on the dots is 7.6 times lower.")
plt.show()

GMRES gains a factor of about 0.10–0.12 per step, against the bound's 0.2, and needs 11 steps where the bound promises 15. The hand curve's actual residual (14 steps to $10^{-10}$) falls faster than its bound at first (0.12 on step 1), since most dots sit closer to 1.25 than the ends do, but its factor creeps up to 0.19 as the dots at the ends of the bunch, where $|1 - z/1.25| = 0.2$, are all that is left. GMRES does better by spreading its roots across the bunch, at 1.02, 1.12, 1.26, 1.39 and 1.48 for $k = 5$, so that no dot is far from a root. Its largest height on the dots is $4.2 \times 10^{-5}$, 7.6 times below the hand curve's $0.2^5 = 3.2 \times 10^{-4}$. The hand curve is lower on the dots right next to 1.25, where its five-fold root flattens it, but GMRES minimises the RMS height over all the dots, and there it wins by the same factor, $1.3 \times 10^{-5}$ against $1.0 \times 10^{-4}$. (The curve with the smallest largest height on an interval is a scaled Chebyshev polynomial, whose roots crowd toward the ends of the interval in the same way.)

**The experiment.** If GMRES can always find a curve at least as good as a hand curve, the question becomes what makes a good curve possible. Keep the width of the bunch fixed at 0.5, and move the bunch: $[a,\ a + 0.5]$ for $a = 0.1$, 1 and 10, with 100 evenly spaced eigenvalues each and $b = (1, \dots, 1)$. Figure 6.4 shows GMRES's $q_3$ for each bunch on one axis, and the three residual histories.

In [ ]:
shifted_bunches = {a: (np.diag(np.linspace(a, a + 0.5, 100)), np.ones(100)) for a in (0.1, 1.0, 10.0)}

fig, axes = plt.subplots(1, 2, figsize=(12, 4.2), constrained_layout=True)
for (a, (A, b)), color in zip(shifted_bunches.items(), SERIES_COLORS):
    label = f"[{a:g}, {a + 0.5:g}]"
    eigenvalues, heights, q, _ = residual_polynomial(A, b, 3)
    plot_residual_polynomial(axes[0], eigenvalues, heights, polynomial=q, z_range=(0.0, 10.6), color=color,
                             label=label)
    b_norm = np.linalg.norm(b)
    _, record = gmres_v4(CountingMatvec(A), b)
    plot_residuals(axes[1], record["residual_history"], b_norm=b_norm, color=color, label=label)
    k = record["iterations"]
    factor = (record["residual_history"][-1] / b_norm) ** (1 / k)
    print(f"bunch {label:12s} |q_3(0) - 1| = {abs(q(0.0) - 1):.1e};   steps to 1e-10: {k:3d},  mean factor per step {factor:.3f}")
axes[0].set_ylim(-0.4, 1.15)
axes[0].set_title(r"GMRES's $q_3$ for each bunch")
axes[0].legend(frameon=False, loc="upper right", title="bunch", fontsize=9, title_fontsize=9)
axes[1].axhline(1e-10, color=REFERENCE_COLOR, linewidth=1)
axes[1].legend(frameon=False, loc="upper right", title="bunch", fontsize=9, title_fontsize=9)
for ax in axes:
    style_axes(ax)
add_figure_caption(fig, "6.4", r"Three bunches of the same width, 0.5, at different distances from 0. Left: GMRES's $q_3$ for each. Right: residual histories, rtol = $10^{-10}$. The bunch at [0.1, 0.6] needs 27 steps; the one at [10, 10.5] needs 6.")
plt.show()

The same width, and very different speeds: 27 steps at a mean factor of 0.42 per step for the bunch nearest 0, 11 steps at 0.10 for $[1, 1.5]$, and 6 steps at 0.013 for $[10, 10.5]$. On the left, the curve for $[10, 10.5]$ has a long way to fall from the pinned point, so it can afford to be nearly flat across the bunch. The curve for $[0.1, 0.6]$ has to fall from 1 to near 0 between $z = 0$ and $z = 0.1$, and it can't then stay small over the whole bunch.

**Rule of thumb: speed is set by the width of the bunch compared with its distance from 0.** Two facts follow from the picture, and Step 8 relies on both.

1. **Scaling $A$ changes nothing.** Replace $A$ by $\alpha A$ for a number $\alpha \ne 0$. The Krylov space $\operatorname{span}\{b, \alpha A b, \dots\}$ is the same space, and $x$ solves $Ax = b$ exactly when $x / \alpha$ solves $(\alpha A)\, x' = b$. So GMRES on $\alpha A$ produces every iterate divided by $\alpha$, and **every residual unchanged**. In the picture the dots move to $\alpha \lambda_i$ and the curve stretches with them, $q(z) \to q(z / \alpha)$: both the width and the distance from 0 are multiplied by $\alpha$.
2. **Shifting $A$ does change convergence.** $A + \sigma I$ moves every dot by $\sigma$ but leaves the pinned point at $z = 0$. Figure 6.4 is exactly that: its three matrices are one bunch shifted by $\sigma = 0$, 0.9 and 9.9.

So a change to $A$ helps only if it moves the dots relative to 0. A preconditioner that just rescales $A$ does nothing. Jacobi preconditioning, which divides each row by its diagonal entry, is such a rescaling on `convdiff_n`, whose diagonal is the constant $2/\Delta s^2$ (Step 8).

A check of fact 1: `gmres_v4` on `diag_narrow` and on $7.3$ times it, 5 steps each.

In [ ]:
x, record = gmres_v4(CountingMatvec(A_narrow), b_narrow, rtol=0.0, max_iterations=5)
x_scaled, record_scaled = gmres_v4(CountingMatvec(7.3 * A_narrow), b_narrow, rtol=0.0, max_iterations=5)
history, history_scaled = np.array(record["residual_history"]), np.array(record_scaled["residual_history"])
print("residual history, A:      ", history)
print("residual history, 7.3 A:  ", history_scaled)
print(f"largest relative difference of the residuals:  {np.max(np.abs(history_scaled - history) / history):.1e}")
print(f"||7.3 x_scaled - x|| / ||x|| = {np.linalg.norm(7.3 * x_scaled - x) / np.linalg.norm(x):.1e}")

### 6.3 Dots near 0 are expensive

`diag_spread`'s 20 dots run from 1 to $10^4$, evenly on a log scale: most of them crowd near the small end, and a few spread out at the large end. A curve held at 1 at $z = 0$ can't be small near $z = 1$ without being steep. Put a root at the smallest dot, $q(z) = 1 - z$, and the height at the largest dot is $1 - 10^4 = -9999$. **Removing a small eigenvalue blows up the large ones.** This is where the condition number enters the picture: the dots reach toward 0 compared with their spread. The best hand curve with a single repeated root puts it at the middle of the range, $(1 - z/c)^k$ with $c = (1 + 10^4)/2$, and that gains only a factor $1 - 2/(1 + 10^4) = 0.9998$ per step on the worst dot.

GMRES does something else: it puts roots on individual dots. Figure 6.5 shows which ones, on a log $z$ axis (which leaves $z = 0$ off the axis), with the residual history next to it.

In [ ]:
b_norm = np.linalg.norm(b_spread)
marked = (2, 5, 8)
fig, axes = plt.subplots(1, 2, figsize=(12, 4.2), constrained_layout=True)
for k in range(1, 9):
    eigenvalues, heights, q, _ = residual_polynomial(A_spread, b_spread, k)
    roots = np.sort(np.real_if_close(q.roots()))
    print(f"k = {k}:  |q_k(0) - 1| = {abs(q(0.0) - 1):.0e},  q_k(1) = {heights[0]:+.3f},  roots of q_k: {np.round(roots, 0)}")
    if k in marked:
        color = SERIES_COLORS[marked.index(k)]
        plot_residual_polynomial(axes[0], eigenvalues, heights, polynomial=q, magnitude=True,
                                 log_eigenvalues=True, color=color, label=f"k = {k}")
print("eigenvalues:", np.round(np.diag(A_spread), 0))
axes[0].set_title(r"$|q_k(z)|$ for diag_spread")
axes[0].legend(frameon=False, loc="lower left", fontsize=9)

_, record = gmres_v4(CountingMatvec(A_spread), b_spread, rtol=0.0)
history = np.array(record["residual_history"]) / b_norm
steps = np.arange(len(history))
ax = axes[1]
plot_residuals(ax, np.maximum(record["residual_history"], bottom * b_norm), b_norm=b_norm, label="GMRES")
for k, color in zip(marked, SERIES_COLORS):
    ax.plot([k], [history[k]], linestyle="none", marker="o", markersize=10, color=color,
            markeredgecolor="white", markeredgewidth=2, zorder=5)
ax.semilogy(steps, (1 - 2 / (1 + 1e4)) ** steps, color=REFERENCE_COLOR, linestyle="--", linewidth=1,
            label=r"best one-root hand curve, $0.9998^k$")
ax.set_ylim(bottom / 2, 3)
ax.legend(frameon=False, loc="lower left", fontsize=9)
style_axes(axes[0])
style_axes(ax, xticks=range(0, 21, 2))
factors = history[1:] / history[:-1]
print("\nGMRES factor per step, k = 1..19:", factors[:19])
add_figure_caption(fig, "6.5", r"diag_spread (eigenvalues 1 to $10^4$, log axis). Left: $|q_k|$ for k = 2, 5, 8. GMRES puts roots on the largest, most isolated eigenvalues first, while $|q|$ stays near 1 on the eigenvalues near 1. Right: the residual (large dots at k = 2, 5, 8) against the best one-root hand curve (0.9998 per step). The zero at k = 20 is drawn on the bottom edge.")
plt.show()

GMRES works from the top down. The single root at $k = 1$ is at 6189, among the largest dots. By $k = 5$ one root sits on the largest dot, $10^4$. By $k = 8$ there are roots on the four largest dots, 2336, 3793, 6158 and $10^4$, while the curve is still at $q_8(1) = 0.98$ on the smallest dot. In Figure 6.5 the spikes at the right-hand end are those roots, and on the left $|q_k|$ stays near 1 over the crowded small dots.

The residual shows the cost: a factor of about 0.94 per step for $k = 4$ to 6, and then faster and faster, down to 0.35 on the last steps before the breakdown at $k = 20$. A rate that improves as the solve goes on is called **superlinear convergence**. In the picture, each dot GMRES has dealt with no longer needs covering, and what is left is a narrower range of dots, closer in relative terms to a bunch.

**Why the large dots go first.** Step 1.7 found that $A^j b$ turns toward the eigenvectors of the largest eigenvalues (power iteration), so those are what $\mathcal{K}_k$ resolves first. They are also cheap to remove: a factor $(1 - z/\lambda_{\text{large}})$ is close to 1 on all the small dots, so it costs nothing there. A factor $(1 - z/\lambda_{\text{small}})$ would cost a lot on all the large dots.

### 6.4 A few bunches take a few steps

On `diag_clustered` the 30 dots sit in three bunches of half-width $10^{-3}$ around 1, 2 and 3. The cubic

$$
q(z) = (1 - z)\Big(1 - \frac{z}{2}\Big)\Big(1 - \frac{z}{3}\Big)
$$

has $q(0) = 1$ and a root at each bunch centre, and within $10^{-3}$ of a root it is at most about $10^{-3}$ times its slope there, which is 1/3 for each of the three bunches. So after 3 steps the residual can be no larger than about $3 \times 10^{-4}$: **one root per bunch, and the residual drops to about the relative width of the bunches after as many steps as there are bunches, whatever $n$ is.** Figure 6.6 shows GMRES's $q_3$, and the residual histories for $n = 30$ and $n = 3000$.

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(12, 4.2), constrained_layout=True)
eigenvalues, heights, q, _ = residual_polynomial(A_clustered, b_clustered, 3)
plot_residual_polynomial(axes[0], eigenvalues, heights, polynomial=q, magnitude=True, label=r"GMRES's $q_3$, n = 30")
axes[0].set_title(r"$|q_3(z)|$ for diag_clustered, n = 30")
axes[0].legend(frameon=False, loc="lower left", fontsize=9)
print(f"n = 30, k = 3:  |q_3(0) - 1| = {abs(q(0.0) - 1):.1e},  roots {np.sort(np.real_if_close(q.roots()))}")

for (n, A, b), color in zip([(30, A_clustered, b_clustered), (3000, A_clustered_large, b_clustered_large)], SERIES_COLORS):
    b_norm = np.linalg.norm(b)
    _, record = gmres_v4(CountingMatvec(A), b)
    history = np.array(record["residual_history"]) / b_norm
    plot_residuals(axes[1], record["residual_history"], b_norm=b_norm, color=color,
                   marker="o" if n == 30 else "x", label=f"n = {n}")
    print(f"n = {n:4d}:  {record['iterations']} steps to 1e-10;  ||r_k|| / ||b|| at k = 3, 6, 9: "
          f"{history[3]:.1e}, {history[6]:.1e}, {history[9]:.1e}")
axes[1].axhline(1e-10, color=REFERENCE_COLOR, linewidth=1)
axes[1].legend(frameon=False, loc="upper right", fontsize=9)
for ax in axes:
    style_axes(ax)
style_axes(axes[1], xticks=range(0, 10))
add_figure_caption(fig, "6.6", r"diag_clustered: three bunches of width $2 \cdot 10^{-3}$ around 1, 2 and 3. Left: $q_3$ has one root in each bunch. Right: at n = 30 and n = 3000 the residual falls in the same staircase, about $10^{-4}$ every 3 steps, and both reach $10^{-10}$ at k = 9.")
plt.show()

GMRES's $q_3$ puts one root in each bunch. The residual after 3 steps is $1.8 \times 10^{-4}$, after 6 steps $3.3 \times 10^{-8}$ and after 9 steps $5.9 \times 10^{-12}$, below the tolerance: a drop of about $10^{-4}$ every 3 steps, the same staircase at $n = 30$ and at $n = 3000$. One hundred times more unknowns, and the same 9 steps.

**Tie to Step 5.** Shrink each bunch to a single point and you get the matrix $\operatorname{diag}(1, 1, 2, 2, 3, 3)$ of section 5.3, where $b$ touches only 3 distinct eigenvalues and GMRES finds the exact answer at $k = 3$, a good breakdown. Clustering is "almost a breakdown": the bunches are almost single eigenvalues, and the residual is almost zero after one step per bunch.

Most matrices are not that tidy. A common shape is one bunch plus a few eigenvalues outside it, the **outliers**. Figure 6.7 takes the bunch $[1, 1.5]$ of `diag_narrow` (100 eigenvalues) on its own, then with three outliers far above it, 5, 20 and 100, then with one outlier at 0.01, close to the pinned point.

In [ ]:
bunch = np.linspace(1.0, 1.5, 100)
outlier_examples = {
    "the bunch [1, 1.5] alone": bunch,
    "with outliers 5, 20, 100": np.concatenate([bunch, [5.0, 20.0, 100.0]]),
    "with an outlier at 0.01": np.concatenate([bunch, [0.01]]),
}
fig, ax = plt.subplots(figsize=(9, 4.2), constrained_layout=True)
for (name, eigenvalues), color in zip(outlier_examples.items(), SERIES_COLORS):
    A, b = np.diag(eigenvalues), np.ones(eigenvalues.size)
    _, record = gmres_v4(CountingMatvec(A), b)
    plot_residuals(ax, record["residual_history"], b_norm=np.linalg.norm(b), color=color, label=name)
    print(f"{name:26s} {record['iterations']:3d} steps to 1e-10")
ax.axhline(1e-10, color=REFERENCE_COLOR, linewidth=1)
ax.legend(frameon=False, loc="lower left", fontsize=9)
style_axes(ax, xticks=range(0, 16))
add_figure_caption(fig, "6.7", r"A bunch [1, 1.5] alone and with outliers, rtol = $10^{-10}$. The three outliers 5, 20 and 100 cost one step each; then the residual falls at the bunch's rate. One outlier at 0.01, close to the pinned point, costs three steps on its own.")
plt.show()

The bunch alone takes 11 steps. With the outliers 5, 20 and 100 it takes 14: GMRES spends one root on each outlier, and then the residual falls at the bunch's rate. A single outlier at 0.01 costs 3 steps on its own. A root at 0.01 makes the curve steep, $1 - z/0.01$ is $-99$ at $z = 1$, and the rest of the curve has to undo that across the bunch (6.3).

This is the shape a good preconditioner aims for (Step 8): most of the dots in a bunch well away from 0, plus a few outliers, preferably not near 0.

### 6.5 A ring around 0

`shift_n` shifts every entry down by one place and wraps the last one around, so applying it 8 times gives back the vector: $A^8 = I$. Its eigenvalues are therefore the 8th roots of unity,

$$
\omega_j = e^{2\pi \mathrm{i}\, j / 8}, \qquad j = 0, 1, \dots, 7,
$$

where $\mathrm{i} = \sqrt{-1}$. They are complex, so the picture moves from an axis to the complex plane: eight dots on the unit circle, a ring of radius 1 around the pinned point $q(0) = 1$. A polynomial held at 1 in the middle of the ring can't be small all the way around it. GMRES's residual is exactly $b$ for $k < 8$ (Step 4.7), so $x_k = 0$ and $q_k \equiv 1$: the best curve is the flat one.

**Why nothing beats the flat curve.** Write a polynomial of degree less than 8 as $q(z) = a_0 + a_1 z + \dots + a_7 z^7$, with $a_0 = q(0) = 1$. Averaged over the 8 roots of unity, each power $\omega_j^m$ with $0 < m < 8$ sums to zero (the points $\omega_j^m$ are spread evenly around the circle), so

$$
\frac{1}{8} \sum_{j=0}^{7} q(\omega_j) = a_0 = q(0) = 1 .
$$

$A$ is orthogonal, and its eigenvectors $u_j$, with entries $u_j[l] = \omega_j^{-(l-1)} / \sqrt{8}$, are orthonormal. Then $b = e_1$ has a component of the same size, $1/\sqrt{8}$, along each one, so $\|q(A)\, b\|_2^2 = \frac18 \sum_j |q(\omega_j)|^2$, the mean of $|q|^2$ over the dots. A mean square is at least the square of the mean, which is 1. So $\|r_k\|_2 \ge \|b\|_2$ for every $k < 8$: **any curve that dips below 1 on some dots must rise above 1 on others.**

Figure 6.8 shows this for one hand curve, with roots on three of the dots:

$$
q(z) = (1 - z)\Big(1 - \frac{z}{\omega_1}\Big)\Big(1 - \frac{z}{\bar\omega_1}\Big) = (1 - z)\big(1 - 2\cos(\pi/4)\, z + z^2\big),
$$

where $\bar\omega_1$ is the complex conjugate of $\omega_1$. Its coefficients are real, and it is 0 at $\omega_0 = 1$, $\omega_1$ and $\omega_7 = \bar\omega_1$.

In [ ]:
n_shift = b_shift.shape[0]
omega = np.exp(2j * np.pi * np.arange(n_shift) / n_shift)   # the eigenvalues omega_j, j = 0, ..., 7
ring_candidate = Polynomial([1.0, -1.0]) * Polynomial([1.0, -2 * np.cos(2 * np.pi / n_shift), 1.0])

eigenvalues_shift = np.linalg.eigvals(A_shift)
print("shift_n's eigenvalues are the 8th roots of unity:",
      np.allclose(np.sort_complex(eigenvalues_shift), np.sort_complex(omega)))
values = ring_candidate(omega)
print(f"hand curve: |q(omega_j)| = {np.abs(values)}")
print(f"            mean of q over the dots = {values.mean():.3f},  RMS of |q| over the dots = {np.sqrt(np.mean(np.abs(values)**2)):.3f}")
_, record = gmres_v4(CountingMatvec(A_shift), b_shift, rtol=0.0)
print(f"gmres_v4's residual history: {np.array(record['residual_history'])}")

fig, axes = plt.subplots(1, 2, figsize=(12, 4.6), constrained_layout=True, gridspec_kw={"width_ratios": [1, 1.3]})
ax = axes[0]
grid = np.linspace(-1.6, 1.6, 401)
X, Y = np.meshgrid(grid, grid)
log_magnitude = np.log10(np.maximum(np.abs(ring_candidate(X + 1j * Y)), 1e-3))
filled = ax.contourf(X, Y, log_magnitude, levels=np.arange(-2.0, 1.51, 0.25), cmap=SEQUENTIAL_CMAP, extend="both")
ax.contour(X, Y, log_magnitude, levels=[0.0], colors=[REFERENCE_COLOR], linewidths=1.5)
fig.colorbar(filled, ax=ax, shrink=0.8, label=r"$\log_{10} |q(z)|$")
theta = np.linspace(0, 2 * np.pi, 400)
ax.plot(np.cos(theta), np.sin(theta), color="white", linewidth=1, linestyle="--")
ax.plot(omega.real, omega.imag, linestyle="none", marker="o", markersize=8, color=SERIES_COLORS[1],
        markeredgecolor="white", markeredgewidth=1.5, label=r"eigenvalues $\omega_j$")
ax.plot([0.0], [0.0], linestyle="none", marker="o", markersize=9, markerfacecolor="white",
        markeredgecolor=REFERENCE_COLOR, markeredgewidth=1.5)
ax.annotate("q(0) = 1", xy=(0.0, 0.0), xytext=(6, 6), textcoords="offset points", fontsize=9)
ax.text(0.78, -1.42, "|q| = 1", color=REFERENCE_COLOR, fontsize=9, ha="center",
        bbox={"facecolor": "white", "edgecolor": "none", "pad": 1.5})
ax.set_aspect("equal")
ax.set_xlabel("Re z")
ax.set_ylabel("Im z")
ax.set_title("hand curve with roots at $\\omega_0, \\omega_1, \\omega_7$")

ax = axes[1]
ax.plot(theta, np.abs(ring_candidate(np.exp(1j * theta))), color=SERIES_COLORS[0], linewidth=2,
        label=r"hand curve: $|q(e^{\mathrm{i}\theta})|$")
ax.plot(2 * np.pi * np.arange(n_shift) / n_shift, np.abs(values), linestyle="none", marker="o", markersize=8,
        color=SERIES_COLORS[1], markeredgecolor="white", markeredgewidth=1.5, label=r"its heights at the $\omega_j$")
ax.axhline(1.0, color=SERIES_COLORS[2], linewidth=2, linestyle="--", label=r"GMRES: $q_k \equiv 1$ for $k \leq 7$")
ax.set_xticks(2 * np.pi * np.arange(n_shift + 1) / n_shift)
ax.set_xticklabels([rf"$\omega_{j}$" if j < n_shift else r"$\omega_0$" for j in range(n_shift + 1)])
ax.set_xlabel(r"angle $\theta$ around the unit circle, $z = e^{\mathrm{i}\theta}$")
ax.set_ylabel(r"$|q(z)|$")
ax.legend(frameon=False, loc="upper left", fontsize=9)
style_axes(ax)
add_figure_caption(fig, "6.8", r"shift_n: the eight eigenvalues on the unit circle around the pinned point $q(0) = 1$ (left, with $|q|$ for a hand curve with roots on three of them). Right: the circle unrolled. The roots pull three dots to 0 and push the others above 1. Averaged over the dots, q is exactly 1, so GMRES's best is $q \equiv 1$, with no progress until k = 8.")
plt.show()

The hand curve is 0 on three dots, but on the other five it rises to between 2 and 6.8. Its mean over the eight dots is exactly 1, as the identity says, and its RMS height is 3.7, so it is worse than doing nothing. GMRES's residual is exactly 1 for $k = 1, \dots, 7$: the flat curve is the best there is. At $k = 8$ the degree is high enough for $q(z) = 1 - z^8$, which is zero on all eight dots, and the residual drops to exactly 0.

**Tie back.** Section 4.7 explained the same stagnation from the vector side: $A \mathcal{K}_k$ is perpendicular to $b$ for $k < 8$, so nothing in $A\mathcal{K}_k$ can reduce $b$. In Step 5's terms, every step is a stagnation step, with rotation pair $(0, \ne 0)$. And it settles the opening table: `shift_n` has the best condition number there is, 1, because its eigenvalues all have size 1, but they surround the pinned point. A milder form of the same problem is a matrix with eigenvalues on both sides of 0: a curve through $(0, 1)$ that is small on both sides has to turn around in between.

**Preview of Step 7.** Restarting GMRES after $m < 8$ steps throws the Krylov space away and starts again from the current residual, which here is still $b$. So GMRES restarted every $m < 8$ steps never converges on `shift_n`.

### 6.6 When the dots mislead (`convdiff_n`)

At the default velocity $c = 20$ on $n = 20$ points, the cell Péclet number is $c\, \Delta s / 2 = 20 / 42 = 0.48$. For a cell Péclet number below 1, the two off-diagonals of `convdiff_n` have the same sign, and its eigenvalues are real:

$$
\lambda_j = \frac{2}{\Delta s^2} \Big( 1 - \sqrt{1 - (c\, \Delta s/2)^2}\, \cos\frac{j\pi}{21} \Big), \qquad j = 1, \dots, 20 .
$$

They lie in one bunch with $\lambda_{\max} / \lambda_{\min} = 14.3$. (That is not the condition number, $\operatorname{cond}(A) = 50.8$: for a matrix that isn't symmetric, the singular values that define $\operatorname{cond}$ differ from the eigenvalues.) By section 6.2, one bunch with that ratio should give a factor well below 0.96 per step. **The experiment: the same dots, a different matrix.** The diagonal twin $\operatorname{diag}(\lambda_1, \dots, \lambda_{20})$ has exactly the eigenvalues of `convdiff_n`; give it $b = (1, \dots, 1)$ too. And as a second experiment, vary $c$: as $c$ grows the square root above shrinks, and the eigenvalues squeeze together toward $2/\Delta s^2$.

In [ ]:
eigenvalues_cd = np.linalg.eigvals(A_cd)
print(f"convdiff_n, c = 20:  largest |imaginary part| of the eigenvalues {np.abs(eigenvalues_cd.imag).max():.1e}")
twin_matrix = np.diag(np.sort(eigenvalues_cd.real))
b_twin = np.ones(20)

fig, axes = plt.subplots(1, 2, figsize=(12, 4.2), constrained_layout=True)
for (name, A, b), color in zip([("convdiff_n, c = 20", A_cd, b_cd), ("its diagonal twin", twin_matrix, b_twin)], SERIES_COLORS):
    b_norm = np.linalg.norm(b)
    _, record = gmres_v4(CountingMatvec(A), b, rtol=0.0)
    history = np.array(record["residual_history"]) / b_norm
    plot_residuals(axes[0], np.maximum(record["residual_history"], bottom * b_norm), b_norm=b_norm, color=color, label=name)
    print(f"{name:20s} ||r_k|| / ||b|| at k = 5: {history[5]:.3f},  at k = 10: {history[10]:.3f}")
axes[0].set_ylim(bottom / 2, 3)
axes[0].set_title("the same eigenvalues, b = (1, ..., 1)")
axes[0].legend(frameon=False, loc="lower left", fontsize=9)

print()
for c, color in zip((10.0, 20.0, 30.0), SERIES_COLORS):
    A, b = convdiff_n(20, c=c)
    eigenvalues = np.linalg.eigvals(A)
    spread = np.abs(eigenvalues).max() / np.abs(eigenvalues).min()
    b_norm = np.linalg.norm(b)
    _, record = gmres_v4(CountingMatvec(A), b, rtol=0.0)
    history = np.array(record["residual_history"]) / b_norm
    plot_residuals(axes[1], np.maximum(record["residual_history"], bottom * b_norm), b_norm=b_norm, color=color,
                   label=rf"c = {c:g}:  $\lambda_{{\max}} / \lambda_{{\min}}$ = {spread:.3g}")
    print(f"c = {c:4g}:  cell Peclet number {c / 42:.2f},  lambda_max / lambda_min = {spread:5.3g},  cond(A) = {np.linalg.cond(A):5.3g},"
          f"  ||r_10|| / ||b|| = {history[10]:.3f}")
axes[1].set_ylim(bottom / 2, 3)
axes[1].set_title("convdiff_n as c grows")
axes[1].legend(frameon=False, loc="lower left", fontsize=9)
for ax in axes:
    style_axes(ax, xticks=range(0, 21, 2))

_, record_twin0 = gmres_v4(CountingMatvec(np.diag(np.sort(np.linalg.eigvalsh(A_cd0)))), b_twin, rtol=0.0, max_iterations=1)
_, record_cd0 = gmres_v4(CountingMatvec(A_cd0), b_cd0, rtol=0.0, max_iterations=1)
print(f"\nc = 0 (symmetric):  ||r_1|| / ||b||  convdiff_n {record_cd0['residual_history'][-1] / np.linalg.norm(b_cd0):.3f},"
      f"  its diagonal twin with b = (1, ..., 1) {record_twin0['residual_history'][-1] / np.linalg.norm(b_twin):.3f}")
add_figure_caption(fig, "6.9", r"Left: convdiff_n (c = 20) and its diagonal twin, a diagonal matrix with the same eigenvalues and b = (1, ..., 1). The twin falls as its eigenvalues predict; convdiff_n crawls at about 0.96 per step. Right: as c grows from 10 to 30 the eigenvalues squeeze from a 49.5:1 spread to 5.5:1, and convdiff_n's residual barely moves. Zeros at k = 20 are drawn on the bottom edge.")
plt.show()

The eigenvalues of `convdiff_n` are real to round-off (the largest imaginary part printed above), so the twin really does have the same dots. After 5 steps the twin's residual is 0.076 and `convdiff_n`'s is 0.836; after 10 steps, 0.006 against 0.670. The same dots, and an 11-fold difference after 5 steps. On the right, the eigenvalue spread falls from 49.5 at $c = 10$ to 14.3 at $c = 20$ and 5.49 at $c = 30$. By section 6.2's rule the squeeze should speed GMRES up. Instead $\|r_{10}\|_2 / \|b\|_2$ goes from 0.604 to 0.670 to 0.693: slightly slower. **On `convdiff_n` the dots don't decide.** Something else does.

**In grid language.** $A$ is tridiagonal: row $i$ couples cell $i$ only to its neighbours $i - 1$ and $i + 1$. So $Av$ spreads a vector by at most one cell, and $A^j b$ by at most $j$ cells. Every iterate $x_k = p(A)\, b$ is built from $b, Ab, \dots, A^{k-1} b$, and its residual from one more power. But $A^{-1}$ couples every cell to every other: the value of the solution at one cell depends on $b$ everywhere (in the differential equation, that is the Green's function). **GMRES has to carry information across the grid one cell per matvec.**

**The exact version, for $b = (1, \dots, 1)$.** Every interior row of $A$ sums to zero: $\frac{2}{\Delta s^2} - \frac{1}{\Delta s^2} - \frac{c}{2\Delta s} - \frac{1}{\Delta s^2} + \frac{c}{2\Delta s} = 0$. So $A \cdot (1, \dots, 1)$ is zero except in cells 1 and 20, where a row is missing a neighbour. Then $A^j b$ is nonzero only within $j$ cells of the two ends, and so is $A x_k$, which is a combination of $Ab, \dots, A^k b$. On the middle $n - 2k$ cells, therefore, $r_k = b - A x_k$ is exactly 1, whatever GMRES does, and

$$
\|r_k\|_2 \ge \sqrt{n - 2k}, \qquad \text{that is,} \qquad \frac{\|r_k\|_2}{\|b\|_2} \ge \sqrt{1 - \frac{2k}{n}} .
$$

The cells that GMRES can clear grow from both ends, one cell per step: two **fronts**. The check below prints where $A^j b$ is nonzero, and $\|r_k\|_2 / \sqrt{n - 2k}$, which the argument says is at least 1. Figure 6.10 shows $A^{-1}$, the residual cell by cell, and the residual history against this floor.

In [ ]:
n = 20
delta_s = 1.0 / (n + 1)
for c, A, b in ((0.0, A_cd0, b_cd0), (20.0, A_cd, b_cd)):
    power = b.copy()
    for j in (1, 2, 3):
        power = A @ power
        support = np.nonzero(np.abs(power) > 1e-12 * np.abs(power).max())[0] + 1
        print(f"c = {c:2g}:  A^{j} b is nonzero on cells {support}")
    ratios = []
    for k in range(1, 10):
        x, _ = gmres_v4(CountingMatvec(A), b, rtol=0.0, max_iterations=k)
        r = b - A @ x
        ratios.append(np.linalg.norm(r) / np.sqrt(n - 2 * k))
        if k == 5:
            print(f"   k = 5:  largest |r_5 - 1| on the middle cells 6..15: {np.abs(r[5:15] - 1).max():.1e}")
    print(f"   ||r_k|| / sqrt(n - 2k) for k = 1..9: {np.array(ratios)}")

A_42, b_42 = convdiff_n(20, c=42.0)
_, record_42 = gmres_v4(CountingMatvec(A_42), b_42, rtol=0.0)
history_42 = np.array(record_42["residual_history"]) / np.linalg.norm(b_42)
k_all = np.arange(len(history_42))
print(f"\nc = 42 (cell Peclet number {42 * delta_s / 2:.3f}):  largest |entry above the diagonal| {np.abs(np.diag(A_42, 1)).max():.1e},"
      f"  diagonal entry {A_42[0, 0]:.0f} = 2/Delta s^2;  largest | ||r_k||/||b|| - sqrt(1 - k/n) | over k = 0..{k_all[-1]}:"
      f" {np.abs(history_42 - np.sqrt(np.maximum(1 - k_all / n, 0))).max():.1e}")

fig, axes = plt.subplots(1, 3, figsize=(15, 4.3), constrained_layout=True, gridspec_kw={"width_ratios": [1.1, 1, 1]})
ax = axes[0]
plot_matrix(ax, np.linalg.inv(A_cd), log=True, title=r"$|A^{-1}|$, c = 20")
band = 5
ax.plot([0.5, n - band], [band + 1, n + 0.5], color=REFERENCE_COLOR, linewidth=1.5)
ax.plot([band + 1, n + 0.5], [0.5, n - band], color=REFERENCE_COLOR, linewidth=1.5)
ax.text(n - 1, 2, "band\n|i - j| ≤ 5", color=REFERENCE_COLOR, fontsize=9, ha="right", va="top")

ax = axes[1]
cells = np.arange(1, n + 1)
for k, color in zip((3, 6, 9), SERIES_COLORS):
    for c, A, b, linestyle in ((20.0, A_cd, b_cd, "-"), (0.0, A_cd0, b_cd0, "--")):
        x, _ = gmres_v4(CountingMatvec(A), b, rtol=0.0, max_iterations=k)
        ax.plot(cells, b - A @ x, color=color, linewidth=2, linestyle=linestyle, marker="o", markersize=4,
                label=f"k = {k}" if linestyle == "-" else None)
ax.plot([], [], color=REFERENCE_COLOR, linestyle="-", label="c = 20")
ax.plot([], [], color=REFERENCE_COLOR, linestyle="--", label="c = 0")
ax.set_xlabel("cell i")
ax.set_ylabel(r"$r_k[i]$")
ax.set_title(r"the residual $r_k$, cell by cell")
ax.set_ylim(-0.3, 1.45)
ax.legend(frameon=False, loc="upper center", fontsize=9, ncols=3)
style_axes(ax, xticks=range(1, n + 1, 2))

ax = axes[2]
k_grid = np.arange(n + 1)
for c, A, b, linestyle in ((20.0, A_cd, b_cd, "-"), (0.0, A_cd0, b_cd0, "--")):
    _, record = gmres_v4(CountingMatvec(A), b, rtol=0.0)
    history = np.array(record["residual_history"]) / np.linalg.norm(b)
    ax.plot(np.arange(len(history)), history, color=SERIES_COLORS[0] if c else SERIES_COLORS[1], linewidth=2,
            linestyle=linestyle, marker="o", markersize=4, label=f"convdiff_n, c = {c:g}")
    if c:
        print(f"c = 20:  mean factor per step over 10 steps {history[10] ** 0.1:.3f},  one-sided curve {(1 - 10 / n) ** 0.05:.3f}")
ax.plot(k_grid, np.sqrt(np.maximum(1 - 2 * k_grid / n, 0)), color=REFERENCE_COLOR, linewidth=1, linestyle=":",
        label=r"two-sided floor $\sqrt{1 - 2k/n}$")
ax.plot(k_grid, np.sqrt(1 - k_grid / n), color=REFERENCE_COLOR, linewidth=1, linestyle="-.",
        label=r"one-sided $\sqrt{1 - k/n}$")
ax.set_xlabel("step k")
ax.set_ylabel(r"$\|r_k\|_2 \,/\, \|b\|_2$")
ax.set_ylim(0, 1.05)
ax.legend(frameon=False, loc="upper right", fontsize=9)
style_axes(ax, xticks=range(0, n + 1, 2))
add_figure_caption(fig, "6.10", r"convdiff_n, b = (1, ..., 1), n = 20. Left: $|A^{-1}|$ (c = 20) is full, so every cell depends on every other, while a degree-5 polynomial in A reaches only the band. Middle: the residual cell by cell at k = 3, 6, 9 (solid c = 20, dashed c = 0): exactly 1 in the middle until the fronts from the two ends meet. With c = 20 only the front from the left-hand end clears cells. Right: $\|r_k\| / \|b\|$ (linear axis) against the two-sided floor $\sqrt{1 - 2k/n}$ and the one-sided curve $\sqrt{1 - k/n}$.")
plt.show()

$A b$ is nonzero only on cells 1 and 20, $A^2 b$ on 1–2 and 19–20, $A^3 b$ on 1–3 and 18–20, at both values of $c$. After 5 steps the middle cells of the residual are exactly 1. And $\|r_k\|_2 / \sqrt{n - 2k}$ is at least 1 at every $k$, as it must be.

**With $c = 0$** the residual sits exactly on the floor: $\|r_k\|_2 = \sqrt{n - 2k}$ for every $k$ (the ratios print as 1.000). GMRES clears one cell per step from each end, and nothing more. At $k = 10$ the two fronts meet in the middle and the residual is 0: that is Step 5's good breakdown at $k = n/2$, seen from the grid.

**With $c = 20$** the floor explains the first step (0.949 against 0.958), and after that the residual follows the **one-sided** curve $\sqrt{1 - k/n}$ instead: 0.836 against 0.866 at $k = 5$, and 0.670 against 0.707 at $k = 10$. The middle panel of Figure 6.10 shows why. Only one front advances: the cells near the left-hand end, $s = 0$, are cleared, one per step, while the right-hand end barely moves. With $c > 0$ the flow carries $u$ to the right, so the left-hand end is the inflow (upwind) boundary. Clearing one cell in 20 per step is a factor of about $\sqrt{1 - 1/20}$ per step, and the mean factor over the first 10 steps is 0.961, against 0.966 for the one-sided curve. **That is the 0.96 of section 4.7**: not a property of the eigenvalues, but of how far one matvec carries information across the grid.

**Nonnormal.** The diagonal twin and `convdiff_n` have the same eigenvalues but different eigenvectors. A matrix whose eigenvectors are perpendicular to each other is called **normal**: diagonal and symmetric matrices are normal, and so is `shift_n`. For a normal matrix $\|q(A)\, b\|_2$ depends only on the heights $q(\lambda_i)$ and on the size of $b$'s component along each eigenvector, so the dots decide. `convdiff_n` with $c > 0$ is **nonnormal**: its eigenvectors are not perpendicular, and the dots alone stop deciding $\|q(A)\, b\|_2$.

With $c = 0$, $A$ is symmetric, so the dots do decide, but only once each is weighted by $b$'s component along its eigenvector. $b = (1, \dots, 1)$ touches only the 10 eigenvectors that are symmetric about the middle of the grid (Step 5.3), so a twin with weight 1 on all 20 eigenvalues is the wrong comparison there: at $k = 1$ the twin is at 0.567 and `convdiff_n` at 0.949. That is why the twin is compared only at $c > 0$, and why the sweep over $c$ on the right of Figure 6.9, which needs no eigenvectors at all, is the second piece of evidence.

**At $c = 42$** the cell Péclet number is exactly 1, and the entry above the diagonal, $-1/\Delta s^2 + c/(2\Delta s)$, is 0 up to round-off ($5.7 \times 10^{-14}$ in floating point, against diagonal entries of 882). So $A$ is lower bidiagonal, with every diagonal entry equal to $2/\Delta s^2 = 882$. Read off the diagonal, its eigenvalues are all the same number: one dot. By section 6.4, one bunch of zero width should take one step. Instead the residual follows $\sqrt{1 - k/n}$ exactly and needs all 20 steps (the printed difference is round-off).

**The default $c$.** $c = 20$ is now `convdiff_n`'s default. It is the value Steps 3–5 and `EXAMPLES` already used, so every number they confirmed still holds; at $n = 20$ its eigenvalues are real and accurate, and the twin separates clearly from it.

### 6.7 The time-step knob

The simulation solves an implicit time step, and the time step sets where the dots sit. The simplest model is the heat equation $u_t = u''$ on $[0, 1]$, with $u(0) = u(1) = 0$. Write $L$ for the matrix of $-u''$ on $n$ interior points, which is `convdiff_n` with $c = 0$, and $\Delta t$ for the time step. One backward-Euler step, $(u^{\text{new}} - u^{\text{old}}) / \Delta t = -L\, u^{\text{new}}$, is the linear system

$$
(I + \Delta t\, L)\, u^{\text{new}} = u^{\text{old}} .
$$

So $A = I + \Delta t\, L$. The eigenvalues of $L$ lie in $(0, 4/\Delta s^2)$, so the dots of $A$ lie in $(1,\ 1 + 4\Delta t / \Delta s^2)$.

- **Small $\Delta t$** gives a tight bunch at 1, far from 0 compared with its width: few steps (6.2).
- **Large $\Delta t$** gives $A \approx \Delta t\, L$. Scaling doesn't matter (6.2), so this is like $L$ itself: dots spread over a ratio of about $4\Delta t / \Delta s^2$, reaching toward 0 compared with their spread (6.3), and a stencil matrix that clears about one cell per step (6.6).

**The proposed rule: the number of steps grows like $\sqrt{\Delta t} / \Delta s$**, the number of cells heat diffuses across in one time step. In one time step heat spreads a distance of about $\sqrt{\Delta t}$, so $A^{-1}$ couples each cell to its neighbours within about $\sqrt{\Delta t} / \Delta s$ cells, and by 6.6 GMRES carries information one cell per matvec. From the eigenvalue side, the best curve on an interval with ratio $\kappa = 1 + 4 \Delta t / \Delta s^2$ (a scaled Chebyshev polynomial, 6.2) needs about $\frac{\sqrt{\kappa}}{2} \ln \frac{2}{\text{rtol}}$ steps, which for large $\Delta t$ is $\ln(2 / \text{rtol}) \cdot \sqrt{\Delta t} / \Delta s \approx 23.7\, \sqrt{\Delta t} / \Delta s$ at $\text{rtol} = 10^{-10}$. The two views agree here because $L$ is symmetric.

**The pitfall.** With $c = 0$, $b = (1, \dots, 1)$ breaks down at $k = n/2$ and $b_i = \sin(\pi s_i)$ at $k = 1$ (Step 5.3, 5.6). $A = I + \Delta t\, L$ has the same eigenvectors as $L$, so the same right-hand sides break down early with it, which would hide the rule. So this section uses $b_i = s_i$ (`rhs="ramp"`), on $n = 400$ points so that the count has room to grow. Figure 6.11 sweeps $\Delta t / \Delta s^2$ from $10^{-2}$ to $10^4$.

In [ ]:
n_heat = 400
L_heat, b_ramp = convdiff_n(n_heat, c=0.0, rhs="ramp")   # L, the matrix of -u''; b_i = s_i
delta_s = 1.0 / (n_heat + 1)


def time_step_matrix(L, dt):
    """A = I + dt L: one backward-Euler step of the heat equation u_t = -L u."""
    return np.eye(L.shape[0]) + dt * L


time_step_ratios = np.array([0.01, 0.03, 0.1, 0.3, 1, 3, 10, 30, 100, 300, 1000, 3000, 10000])   # Delta t / Delta s^2
step_counts = {}
print(f"{'dt/ds^2':>8s} {'sqrt(dt)/ds':>11s} {'steps':>6s} {'steps / (sqrt(dt)/ds)':>22s} {'converged':>9s} {'breakdown':>9s}")
for ratio in time_step_ratios:
    _, record = gmres_v4(CountingMatvec(time_step_matrix(L_heat, ratio * delta_s**2)), b_ramp)
    step_counts[ratio] = record["iterations"]
    print(f"{ratio:8g} {np.sqrt(ratio):11.2f} {record['iterations']:6d} {record['iterations'] / np.sqrt(ratio):22.1f}"
          f" {record['converged']!s:>9s} {record['breakdown']!s:>9s}")

counts = np.array([step_counts[ratio] for ratio in time_step_ratios])
middle = (time_step_ratios >= 1) & (time_step_ratios <= 300)
C = np.exp(np.mean(np.log(counts[middle] / np.sqrt(time_step_ratios[middle]))))
print(f"\nfitted over 1 <= dt/ds^2 <= 300:  steps = C sqrt(dt)/ds with C = {C:.1f};  "
      f"the estimate's ln(2/rtol) = {np.log(2e10):.1f}")

fig, ax = plt.subplots(figsize=(9, 4.4), constrained_layout=True)
ax.loglog(time_step_ratios, counts, color=SERIES_COLORS[0], linewidth=2, marker="o", markersize=6,
          label="steps to rtol = 1e-10")
dense = np.geomspace(0.3, 1e4, 200)
ax.loglog(dense, C * np.sqrt(dense), color=REFERENCE_COLOR, linewidth=1, linestyle="--",
          label=rf"{C:.1f} $\sqrt{{\Delta t}} / \Delta s$")
ax.axhline(n_heat, color=REFERENCE_COLOR, linewidth=1, linestyle=":")
ax.text(0.012, n_heat * 1.12, f"n = {n_heat}: the most any solve can take", color=REFERENCE_COLOR, fontsize=9)
ax.set_xlabel(r"$\Delta t / \Delta s^2$")
ax.set_ylabel("steps")
ax.set_ylim(3, 900)
ax.legend(frameon=False, loc="lower right", fontsize=9)
style_axes(ax)
add_figure_caption(fig, "6.11", "Steps to rtol = 1e-10 for A = I + Δt L (n = 400), against Δt/Δs². Small Δt gives a tight bunch at 1 and few steps. From Δt/Δs² ≈ 1 the count grows like √Δt/Δs, about 22 steps per cell crossed, until it reaches n.")
plt.show()

In [ ]:
middle_row = n_heat // 2
inverses = {ratio: np.linalg.inv(time_step_matrix(L_heat, ratio * delta_s**2)) for ratio in (10, 1000)}
largest = max(np.abs(M).max() for M in inverses.values())
fig, axes = plt.subplots(1, 2, figsize=(12, 5.0), constrained_layout=True)
for ax, (ratio, M) in zip(axes, inverses.items()):
    row = np.abs(M[middle_row])
    distance = 40
    decay_length = distance / np.log(row[middle_row] / row[middle_row + distance])   # cells per factor e
    print(f"dt/ds^2 = {ratio:5g}:  row {middle_row + 1} of A^-1 falls by a factor e every {decay_length:.2f} cells"
          f" (measured over {distance} cells);  sqrt(dt)/ds = {np.sqrt(ratio):.2f}")
    image = plot_matrix(ax, M, log=True, floor=1e-8 * largest / np.abs(M).max(), vmin=1e-8 * largest, vmax=largest,
                        colorbar=False, title=rf"$|A^{{-1}}|$, $\Delta t / \Delta s^2$ = {ratio:g}")
fig.colorbar(image, ax=axes, shrink=0.8, label="|entry|")
add_figure_caption(fig, "6.12", r"$|A^{-1}|$ for $A = I + \Delta t\, L$ at $\Delta t / \Delta s^2$ = 10 (left) and 1000 (right), n = 400, on a shared log scale (blank: below $10^{-8}$ of the largest entry). Away from the diagonal each row falls by a factor e every $\sqrt{\Delta t} / \Delta s$ cells: 3.2 and 31.6.")
plt.show()

Below $\Delta t / \Delta s^2 = 1$ the dots form a tight bunch at 1 and GMRES needs 5 to 13 steps. From $\Delta t / \Delta s^2 = 1$ to 300 the count is 23, 39, 70, 123, 226 and 377 steps, about 22 steps for every cell crossed, $\sqrt{\Delta t} / \Delta s$, close to the estimate 23.7 above. From $\Delta t / \Delta s^2 = 1000$ the count hits the cap: GMRES needs all $n = 400$ steps, and at $k = n$ the Krylov space is everything. Figure 6.12 shows the same scale in $A^{-1}$: away from the diagonal, each row falls by a factor $e$ every 3.18 cells at $\Delta t / \Delta s^2 = 10$ and every 31.6 cells at 1000, which is $\sqrt{\Delta t} / \Delta s$ (3.16 and 31.6).

**Tie to production.** The simulation's implicit solver converges in few GMRES steps when $\Delta t$ is moderate: few cells crossed per time step. Refining the grid at a fixed $\Delta t$ makes $\Delta s$ smaller, so $\sqrt{\Delta t} / \Delta s$ grows, and the step count with it. Today `preconditioner.py` registers only `"none"`, so production relies on this knob alone. Step 8 adds the other one.

### 6.8 Why many steps hurt

**Work per step.** Step $j$ of `gmres_v4` orthogonalises the new vector $A v_j$ against the $j$ basis vectors $v_1, \dots, v_j$ with classical Gram–Schmidt applied twice (Step 2). Each pass is two products with the $n \times j$ basis, $V_j^\mathsf{T} w$ and $V_j h$, of $nj$ multiply-adds each, so a step costs about $4nj$ multiply-adds, and the solver has to store $j + 1$ vectors of length $n$. The matvec itself, for a stencil like `convdiff_n`, costs about $3n$: three entries per row.

**In total**, $k$ steps cost about $\sum_{j=1}^{k} 4nj \approx 2nk^2$ multiply-adds for the orthogonalisation, a triangle that grows with the square of the step count, and $(k+1)\,n$ numbers of memory, its height. (The tutorial's dense `CountingMatvec` costs $n^2$ per matvec, but it stands in for production's function operator, which is a stencil.) The work is counted, not timed: a time changes on every run, and the repository strips notebook outputs anyway. Section 4.8 already timed one part of a step.

Figure 6.13 counts it for the $\Delta t / \Delta s^2 = 300$ solve of 6.7.

In [ ]:
ratio = 300
k = step_counts[ratio]
j = np.arange(1, k + 1)
orthogonalisation = 4 * n_heat * j              # step j: two passes of two products with the n x j basis
stencil = np.full(k, 3 * n_heat)               # step j: one stencil matvec
basis_memory = (j + 1) * n_heat                # numbers stored after step j
print(f"dt/ds^2 = {ratio}: {k} steps, n = {n_heat}")
print(f"  total multiply-adds: orthogonalisation {orthogonalisation.sum():.2e},  stencil matvecs {stencil.sum():.2e}"
      f"  (ratio {orthogonalisation.sum() / stencil.sum():.0f})")
print(f"  memory: the basis {basis_memory[-1]:,} numbers,  the stencil {3 * n_heat:,} numbers")

fig, axes = plt.subplots(1, 3, figsize=(15, 4.0), constrained_layout=True)
ax = axes[0]
ax.plot(j, orthogonalisation, color=SERIES_COLORS[0], linewidth=2, label=r"orthogonalisation, $4nj$")
ax.plot(j, stencil, color=SERIES_COLORS[1], linewidth=2, label=r"stencil matvec, $3n$")
ax.set_xlabel("step j")
ax.set_ylabel("multiply-adds in step j")
ax.set_title("work per step")
ax.legend(frameon=False, loc="upper left", fontsize=9)
ax = axes[1]
ax.plot(j, np.cumsum(orthogonalisation), color=SERIES_COLORS[0], linewidth=2, label=r"orthogonalisation, $\approx 2nk^2$")
ax.plot(j, np.cumsum(stencil), color=SERIES_COLORS[1], linewidth=2, label=r"stencil matvecs, $3nk$")
ax.set_xlabel("step k")
ax.set_ylabel("multiply-adds in steps 1..k")
ax.set_title("total work")
ax.legend(frameon=False, loc="upper left", fontsize=9)
ax = axes[2]
ax.plot(j, basis_memory, color=SERIES_COLORS[0], linewidth=2, label=r"the basis, $(k+1)\,n$")
ax.axhline(3 * n_heat, color=SERIES_COLORS[1], linewidth=2, label=r"the stencil, $3n$")
ax.set_xlabel("step k")
ax.set_ylabel("numbers stored")
ax.set_title("memory")
ax.legend(frameon=False, loc="upper left", fontsize=9)
for ax in axes:
    style_axes(ax)
add_figure_caption(fig, "6.13", r"One GMRES solve of $I + \Delta t\, L$ (n = 400, $\Delta t / \Delta s^2$ = 300, 377 steps), counted rather than timed. Left: step j's orthogonalisation (4nj multiply-adds) against its stencil matvec (3n). Middle: the total work of the orthogonalisation grows like $2nk^2$. Right: basis memory grows like $(k + 1)\,n$.")
plt.show()

Over the 377 steps the orthogonalisation costs $1.1 \times 10^8$ multiply-adds and the stencil matvecs $4.5 \times 10^5$: 250 times less. From step 1 on, orthogonalising costs more than the matvec, and by the last step it costs 500 times more. The basis holds 151,200 numbers, against 1,200 for the stencil. In the simulation $n$ is the number of unknowns of the whole grid, so the memory, more than the work, is what runs out first.

The slow cases of this step, dots reaching toward 0 (6.3), a ring around 0 (6.5), a stencil matrix clearing one cell per step (6.6) and a large $\Delta t$ (6.7), all mean many steps, and many steps mean a growing triangle of work and a growing basis. There are two ways out. Cap the triangle: stop after $m$ steps, keep the iterate, and start again from its residual (restarting, Step 7). Or need fewer steps: change the problem so that its dots move into bunches away from 0 (preconditioning, Step 8).

### Summary of Step 6

- **The residual is a curve through $(0, 1)$.** $r_k = q_k(A)\, b$ with $\deg q_k \le k$ and $q_k(0) = 1$, and GMRES picks the $q_k$ that makes $\|q_k(A)\, b\|_2$ smallest.
- For a diagonal $A$ and $b = (1, \dots, 1)$, **$\|r_k\|_2 / \|b\|_2$ is the RMS height of the dots** $q_k(\lambda_i)$. `residual_polynomial` reads $q_k$ off the residual, with $q_k(0) \approx 1$ as its check.
- **GMRES beats any hand curve**: $\|r_k\|_2 \le \max_i |q(\lambda_i)|\, \|b\|_2$ for every $q$ of degree at most $k$ with $q(0) = 1$. On `diag_narrow` it gains 0.10–0.12 per step against the hand curve's bound of 0.2, by spreading its roots across the bunch.
- **Speed is set by the width of the bunch against its distance from 0.** Scaling $A$ leaves every residual unchanged; shifting $A$ moves the dots relative to the pinned point and does change convergence.
- **Dots near 0 are expensive**, and GMRES removes the large, isolated dots first (`diag_spread`), which gives superlinear convergence.
- **One root per bunch**: three tight bunches take about 3 steps per factor $10^{-4}$, at any $n$. Outliers far from 0 cost about one step each; an outlier near 0 costs more.
- **A ring around 0 is hopeless**: on `shift_n` the mean of any $q$ over the dots is $q(0) = 1$, so GMRES makes no progress until $k = n$, even with condition number 1.
- **On stencil matrices the residual clears about one cell per step**, and the dots can mislead. `convdiff_n` with $c = 20$ follows $\sqrt{1 - k/n}$, a factor of 0.96 per step, while a diagonal matrix with the same eigenvalues is 11 times faster after 5 steps. Its eigenvectors are not perpendicular: it is **nonnormal**.
- **The time step sets the spread**: for $A = I + \Delta t\, L$ the step count grows like $\sqrt{\Delta t} / \Delta s$, about 22 steps per cell crossed at $\text{rtol} = 10^{-10}$, up to $n$.
- **Many steps cost about $2nk^2$ work and $(k + 1)\, n$ memory**, far more than the stencil matvecs themselves.

**Ties to code.** No production code changes in this step. Production's `gmres.py: gmres` has two arguments that this step motivates, and Steps 7 and 8 explain them: `restart`, which caps the triangle of 6.8 at $m$ steps per cycle, and `preconditioner`, which moves the dots. Today `preconditioner.py` registers only `"none"`, so in production the only lever on the step count is the time step of 6.7.

**Next: Step 7.** Many steps are expensive: step $j$ orthogonalises against $j$ vectors and the solver stores all of them. Step 7 caps the number of steps at $m$, keeps the iterate, and starts again from its residual. In the picture, the residual polynomial becomes a product of curves of degree $m$, each held at $(0, 1)$, which is cheaper but can be much worse: on `shift_n`, with $m < 8$, it never converges.